# Catalog Population

Populate the existing LWI Region 3 StormHub catalog with AORC storm-event data. This notebook assumes the base catalog has already been created by `catalog_creation.ipynb`.

In [ ]:
from datetime import datetime
from pathlib import Path

from stormhub.logger import initialize_logger
from stormhub.met.storm_catalog import (
    StormCatalog,
    add_storm_dss_files,
    create_normal_precip,
    new_collection,
    resume_collection,
    stac_to_parquet,
)
from stormhub.met.zarr_to_dss import NOAADataVariable

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

CATALOG_ID = "lwi-region3"
CATALOG_DIR = REPO_ROOT / "catalogs" / CATALOG_ID
CATALOG_FILE = CATALOG_DIR / "catalog.json"

CATALOG_FILE

## Population Parameters

`SMOKE_TEST = True` limits the run to the dates in `SPECIFIC_DATES`. Set it to `False` for the full configured date range.

In [ ]:
# All Collection Args
START_DATE = "1979-02-01"
END_DATE = "2024-12-31"
TOP_N_EVENTS = 440

# Collection Args
STORM_DURATION_HOURS = 48
MIN_PRECIP_THRESHOLD = 2.5
CHECK_EVERY_N_HOURS = 6
NUM_WORKERS = 8
USE_THREADS = False  # Windows: keep this False.

# Development controls
SMOKE_TEST = True
SPECIFIC_DATES = [datetime(2020, 6, 1, 0), datetime(2020, 8, 27, 0)]
WITH_TRACEBACK = True
CREATE_NEW_ITEMS = True

# Optional derived products
RUN_DSS_EXPORT = False
RUN_NORMAL_PRECIP = False
RUN_GEOPARQUET_EXPORT = True

params = {
    "start_date": START_DATE,
    "end_date": END_DATE,
    "top_n_events": TOP_N_EVENTS,
    "storm_duration_hours": STORM_DURATION_HOURS,
    "min_precip_threshold": MIN_PRECIP_THRESHOLD,
    "check_every_n_hours": CHECK_EVERY_N_HOURS,
    "num_workers": NUM_WORKERS,
    "use_threads": USE_THREADS,
    "smoke_test": SMOKE_TEST,
}
params

## Input Rundown

| Input | Purpose |
| --- | --- |
| `START_DATE`, `END_DATE` | Inclusive period used to generate candidate AORC storm start times. |
| `TOP_N_EVENTS` | Number of ranked storms retained as STAC items after filtering and ranking. |
| `STORM_DURATION_HOURS` | Rolling precipitation accumulation window. Also determines the collection id, such as `48hr-events`. |
| `MIN_PRECIP_THRESHOLD` | Minimum event precipitation threshold used before ranking storms. |
| `CHECK_EVERY_N_HOURS` | Candidate start-time interval. Smaller values search more densely and cost more. |
| `NUM_WORKERS` | Parallel workers used while collecting event stats and creating items. |
| `USE_THREADS` | Executor mode. Use `False` on native Windows per the StormHub guidance. |
| `SPECIFIC_DATES` | Optional list of exact candidate start datetimes. Useful for smoke tests or targeted reruns. |
| `CREATE_NEW_ITEMS` | If `True`, writes item JSON for selected storms; if `False`, updates ranking metadata on existing items. |
| `WITH_TRACEBACK` | Includes tracebacks in error logging for easier debugging. |
| `RUN_DSS_EXPORT` | Adds DSS assets to storm items. This can be expensive and requires `hecdss`. |
| `RUN_NORMAL_PRECIP` | Builds an annual-max/normal precipitation GeoTIFF. This is also a large AORC workload. |
| `RUN_GEOPARQUET_EXPORT` | Converts populated STAC items to GeoParquet for easier analysis and sharing. |

## Load Catalog

In [ ]:
if not CATALOG_FILE.exists():
    raise FileNotFoundError(f"Create the base catalog first: {CATALOG_FILE}")

initialize_logger()
storm_catalog = StormCatalog.from_file(str(CATALOG_FILE))
storm_catalog.id, storm_catalog.spm.catalog_file

## Create Or Update Event Collection

For the full run, set `SMOKE_TEST = False` in the parameter cell and rerun from there.

In [ ]:
specific_dates = SPECIFIC_DATES if SMOKE_TEST else None

storm_collection = new_collection(
    storm_catalog,
    start_date=START_DATE,
    end_date=END_DATE,
    storm_duration=STORM_DURATION_HOURS,
    min_precip_threshold=MIN_PRECIP_THRESHOLD,
    top_n_events=TOP_N_EVENTS,
    check_every_n_hours=CHECK_EVERY_N_HOURS,
    specific_dates=specific_dates,
    use_threads=USE_THREADS,
    num_workers=NUM_WORKERS,
    with_tb=WITH_TRACEBACK,
    create_new_items=CREATE_NEW_ITEMS,
)

storm_collection.id if storm_collection else None

## Resume Missing Dates

Use this if a long run stops partway through. It searches the existing `storm-stats.csv` and processes only missing candidate dates.

In [ ]:
# Uncomment to resume a partial full run.
# resume_collection(
#     catalog=str(CATALOG_FILE),
#     start_date=START_DATE,
#     end_date=END_DATE,
#     storm_duration=STORM_DURATION_HOURS,
#     min_precip_threshold=MIN_PRECIP_THRESHOLD,
#     top_n_events=TOP_N_EVENTS,
#     check_every_n_hours=CHECK_EVERY_N_HOURS,
#     num_workers=NUM_WORKERS,
#     with_tb=WITH_TRACEBACK,
#     create_items=CREATE_NEW_ITEMS,
#     use_threads=USE_THREADS,
# )

## Optional DSS Assets

DSS export writes meteorological grids for each selected event. Leave this off until the event collection looks right.

In [ ]:
if RUN_DSS_EXPORT:
    add_storm_dss_files(
        storm_catalog,
        aoi_name=CATALOG_ID,
        use_valid_region=False,
        variable_duration_map={NOAADataVariable.APCP: STORM_DURATION_HOURS},
        dss_output_dir=str(CATALOG_DIR / "dss"),
        output_resolution_km=1,
    )
else:
    print("RUN_DSS_EXPORT is False; skipping DSS export.")

## Optional Normal Precipitation Grid

This computes annual maximum grids and averages them into a normal precipitation GeoTIFF. Consider testing a short year range before running the full 1980-2024 period.

In [ ]:
if RUN_NORMAL_PRECIP:
    normal_dir = CATALOG_DIR / "normal_precip"
    normal_dir.mkdir(parents=True, exist_ok=True)

    create_normal_precip(
        start_year=1980,
        end_year=2024,
        catalog=storm_catalog,
        storm_duration_hours=STORM_DURATION_HOURS,
        every_n_hours=24,
        months=None,
        ams_zarr_path=str(normal_dir / "ams_grids.zarr"),
        normal_precip_grid_path=str(normal_dir / "normalized_precip.tif"),
    )
else:
    print("RUN_NORMAL_PRECIP is False; skipping normal precipitation grid.")

## Optional GeoParquet Export

GeoParquet gives you a compact table for EDA after the STAC item JSON has been created.

In [ ]:
if RUN_GEOPARQUET_EXPORT and storm_collection is not None:
    parquet_path = CATALOG_DIR / storm_collection.id / "all-items.parquet"
    stac_to_parquet(storm_collection, parquet_file=str(parquet_path))
    print(parquet_path)
else:
    print("Skipping GeoParquet export.")

## Serve The Populated Catalog

Run this from a PowerShell terminal with the `stormhub` environment active:

```powershell
stormhub-server .\catalogs\lwi-region3 127.0.0.1 5000
```

Then open `http://localhost:5000/catalog.json` or the STAC Browser link shown by the directory listing.